# CIFAR-10 Lift Test Experiment

This notebook demonstrates the CIFAR-10 Lift Test for evaluating 15 neural network architectures across 4 groups:

- **Group 1**: Thermogeometric Optimal (4 architectures) - optimal thermogeometric design
- **Group 2**: Topology Destroyer (4 architectures) - same params but with 8x bottlenecks
- **Group 3**: Thermal Boiling Furnace (4 architectures) - ReLU ablation baseline
- **Group 4**: Traditional Baselines (3 architectures) - ResNet-18, VGG-11, DenseNet-40

## Quick Start

```python
from experiments.lift_test import train, evaluate

# Phase A: Train all 15 architectures for 30 epochs
results = train.train_phase_a()

# Phase B: Train selected architectures for 150 epochs
phase_b_results = train.train_phase_b(results)

# Evaluate and visualize results
metrics = evaluate.load_results('./experiments/lift_test/results')
evaluate.print_summary(metrics)
```

In [ ]:
# Setup and imports
import sys
sys.path.insert(0, '../..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from experiments.lift_test import (
    list_models, get_model, get_model_info,
    train, evaluate, ArchitectureMetrics
)

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

## 1. List All Architectures

In [ ]:
# List all 15 architectures
archs = list_models()
print(f"Total architectures: {len(archs)}\n")

for i, name in enumerate(archs, 1):
    info = get_model_info(name)
    print(f"{i:2d}. {name:20s} | Group {info['group']} | "
          f"Params: {info['parameters']:,} | FLOPs: {info['flops']:,}")

## 2. Model Information Summary

In [ ]:
# Create summary DataFrame
import pandas as pd

model_info = []
for name in list_models():
    info = get_model_info(name)
    model_info.append({
        'Architecture': name,
        'Group': info['group'],
        'Parameters': info['parameters'],
        'FLOPs': info['flops'],
        'Activation': info['activation'],
        'Skip Conn.': info['has_skip'],
        'LayerNorm': info['has_norm']
    })

df = pd.DataFrame(model_info)
df = df.sort_values(['Group', 'Architecture'])
df

## 3. Visualize Model Complexity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Parameters vs FLOPs
ax1 = axes[0]
colors = {1: 'blue', 2: 'red', 3: 'orange', 4: 'green'}
group_names = {1: 'ThermoNet', 2: 'ThermoBot', 3: 'ReLUFurnace', 4: 'Traditional'}

for group in [1, 2, 3, 4]:
    mask = df['Group'] == group
    ax1.scatter(df[mask]['Parameters']/1e6, df[mask]['FLOPs']/1e9, 
                c=colors[group], label=group_names[group], s=100, alpha=0.7)

ax1.set_xlabel('Parameters (Millions)')
ax1.set_ylabel('FLOPs (Billions)')
ax1.set_title('Model Complexity by Architecture')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Bar chart of FLOPs
ax2 = axes[1]
df_sorted = df.sort_values('FLOPs', ascending=True)
bars = ax2.barh(df_sorted['Architecture'], df_sorted['FLOPs']/1e9, 
                color=[colors[g] for g in df_sorted['Group']])
ax2.set_xlabel('FLOPs (Billions)')
ax2.set_title('FLOPs by Architecture')
ax2.grid(True, alpha=0.3, axis='x')

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colors[g], label=group_names[g]) for g in [1, 2, 3, 4]]
ax2.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

## 4. Training Functions

In [ ]:
# Phase A: Train all 15 architectures
# This will take significant time depending on hardware

# results_a = train.train_phase_a(
#     output_dir='./results',
#     device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# )

In [ ]:
# Phase B: Train selected architectures for longer
# phase_b_archs = ['ThermoNet-3', 'ThermoNet-7', 'ReLUFurnace-3', 'ReLUFurnace-7', 'ResNet-18-CIFAR']
# 
# results_b = train.train_phase_b(
#     phase_a_results=results_a,
#     output_dir='./results',
#     device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
#     architectures=phase_b_archs
# )

## 5. Load and Analyze Results

In [ ]:
# Load results from disk
metrics_list = evaluate.load_results('./results')

print(f"Loaded results for {len(metrics_list)} architectures")

# Display summary
evaluate.print_summary(metrics_list)

## 6. Results Comparison Table

In [ ]:
# Generate comparison table
print(evaluate.generate_comparison_table(metrics_list, sort_by='test_acc'))

## 7. Visualize Results

In [ ]:
if metrics_list:
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # Prepare data
    names = [m.name for m in metrics_list]
    test_accs = [m.test_acc for m in metrics_list]
    alphas = [m.alpha * 1000 for m in metrics_list]  # Scale for display
    gen_gaps = [m.generalization_gap for m in metrics_list]
    group_colors = [colors[m.group] for m in metrics_list]
    
    # Sort by test accuracy
    sorted_indices = np.argsort(test_accs)[::-1]
    names = [names[i] for i in sorted_indices]
    test_accs = [test_accs[i] for i in sorted_indices]
    alphas = [alphas[i] for i in sorted_indices]
    gen_gaps = [gen_gaps[i] for i in sorted_indices]
    group_colors = [group_colors[i] for i in sorted_indices]
    
    # 1. Test Accuracy by Architecture
    ax1 = axes[0, 0]
    bars = ax1.bar(names, test_accs, color=group_colors)
    ax1.set_ylabel('Test Accuracy (%)')
    ax1.set_title('Test Accuracy by Architecture')
    ax1.tick_params(axis='x', rotation=45, ha='right')
    ax1.set_ylim([0, 100])
    ax1.grid(True, alpha=0.3, axis='y')
    
    # 2. α (Thermal Conductivity Proxy) by Architecture
    ax2 = axes[0, 1]
    bars = ax2.bar(names, alphas, color=group_colors)
    ax2.set_ylabel('α × 1000')
    ax2.set_title('α (Accuracy / log(FLOPs)) by Architecture')
    ax2.tick_params(axis='x', rotation=45, ha='right')
    ax2.grid(True, alpha=0.3, axis='y')
    
    # 3. Generalization Gap
    ax3 = axes[1, 0]
    bars = ax3.bar(names, gen_gaps, color=group_colors)
    ax3.set_ylabel('Generalization Gap (%)')
    ax3.set_title('Generalization Gap (Train Acc - Test Acc)')
    ax3.tick_params(axis='x', rotation=45, ha='right')
    ax3.grid(True, alpha=0.3, axis='y')
    
    # 4. Group Summary
    ax4 = axes[1, 1]
    group_means = {}
    for g in [1, 2, 3, 4]:
        group_accs = [m.test_acc for m in metrics_list if m.group == g]
        group_means[g] = np.mean(group_accs) if group_accs else 0
    
    groups = [group_names[g] for g in [1, 2, 3, 4]]
    means = [group_means[g] for g in [1, 2, 3, 4]]
    bars = ax4.bar(groups, means, color=[colors[g] for g in [1, 2, 3, 4]])
    ax4.set_ylabel('Mean Test Accuracy (%)')
    ax4.set_title('Mean Test Accuracy by Group')
    ax4.set_ylim([0, 100])
    ax4.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar, val in zip(bars, means):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.1f}%', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

## 8. Correlation Analysis

In [ ]:
if metrics_list:
    correlations = evaluate.correlation_analysis(metrics_list)
    
    print("Correlation Analysis:\n")
    for key, vals in correlations.items():
        print(f"{key}:")
        for metric, value in vals.items():
            print(f"  {metric}: {value:.4f}")
        print()

## 9. Grokking Analysis

In [ ]:
if metrics_list:
    grokking_metrics = [m for m in metrics_list if m.grokking_epoch is not None]
    
    print(f"Architectures showing grokking: {len(grokking_metrics)}/{len(metrics_list)}")
    print("\nGrokking Details:\n")
    
    for m in sorted(grokking_metrics, key=lambda x: x.grokking_epoch):
        print(f"  {m.name:20s}: Epoch {m.grokking_epoch} "
              f"(Improvement: {m.grokking_improvement*100:.1f}%)")

## 10. Phase A vs Phase B Comparison (if available)

In [ ]:
# Compare Phase A and Phase B results for architectures trained in both
if metrics_list:
    phase_a = {m.name: m for m in metrics_list if m.phase == 'A'}
    phase_b = {m.name: m for m in metrics_list if m.phase == 'B'}
    
    common_archs = set(phase_a.keys()) & set(phase_b.keys())
    
    if common_archs:
        print(f"Architectures with both Phase A and B results: {len(common_archs)}")
        print("\nImprovement from Phase B:\n")
        
        for name in sorted(common_archs):
            a = phase_a[name]
            b = phase_b[name]
            improvement = b.test_acc - a.test_acc
            print(f"  {name:20s}: {a.test_acc:.2f}% -> {b.test_acc:.2f}% "
                  f"({improvement:+.2f}%)")
    else:
        print("No architectures trained in both phases yet.")

---

## Summary

This notebook provides a comprehensive analysis of the CIFAR-10 Lift Test results. Key metrics to observe:

1. **Test Accuracy**: Primary metric for model performance
2. **α (alpha)**: Thermal conductivity proxy = accuracy / log(FLOPs)
3. **Generalization Gap**: Difference between train and test accuracy
4. **Grokking Epoch**: When significant improvement occurs during training

The correlation between α and actual test accuracy provides insight into whether the thermogeometric theory predicts network performance.